# ULTOSC Signal Strategy - Research Notebook

Defines Ultimate Oscillator (ULTOSC) indicator math and signal logic for parity with Pine Script and engine strategies.

## Indicator Math

### Buying Pressure (BP) and True Range (TR)
```
prev_close = nz(close[1], close)  # Bar 0: use close
BP = close - min(low, prev_close)
TR = max(high, prev_close) - min(low, prev_close)
```

### Ultimate Oscillator
```
avgN = sum(BP, N) / sum(TR, N)  # N-period average
UO = 100 * (4*avg1 + 2*avg2 + avg3) / 7
```

### First-Bar Handling (Pine)
- Bar 0: `prev_close = close` (nz(close[1], close))
- BP and TR are computed on bar 0
- First UO value: bar_index >= timeperiod3

In [ ]:
import numpy as np
import pandas as pd
import talib
import yfinance as yf
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Load sample data
ticker = yf.Ticker("ETH-USD")
df = ticker.history(interval="1h", start="2025-01-01", end="2025-02-15")
df = df.reset_index()
df.columns = [c.replace(" ", "_") for c in df.columns]
high = df["High"].values.astype(np.float64)
low = df["Low"].values.astype(np.float64)
close = df["Close"].values.astype(np.float64)
print(f"Loaded {len(df)} bars")

In [ ]:
# ULTOSC with TA-Lib (t1=14, t2=28, t3=36)
t1, t2, t3 = 14, 28, 36
ultosc_talib = talib.ULTOSC(high, low, close, t1, t2, t3)
df["ULTOSC_TA-Lib"] = ultosc_talib

In [ ]:
def ultosc_handrolled(high, low, close, t1, t2, t3):
    """Hand-rolled ULTOSC with bar-0 handling (prev_close=close)."""
    n = len(close)
    prev_close = np.roll(close, 1)
    prev_close[0] = close[0]  # Pine: nz(close[1], close)
    
    bp = close - np.minimum(low, prev_close)
    tr = np.maximum(high, prev_close) - np.minimum(low, prev_close)
    
    bp1 = np.convolve(bp, np.ones(t1), "valid")
    tr1 = np.convolve(tr, np.ones(t1), "valid")
    bp2 = np.convolve(bp, np.ones(t2), "valid")
    tr2 = np.convolve(tr, np.ones(t2), "valid")
    bp3 = np.convolve(bp, np.ones(t3), "valid")
    tr3 = np.convolve(tr, np.ones(t3), "valid")
    
    avg1 = np.where(tr1 != 0, bp1 / tr1, 0)
    avg2 = np.where(tr2 != 0, bp2 / tr2, 0)
    avg3 = np.where(tr3 != 0, bp3 / tr3, 0)
    
    uo = 100 * (4*avg1 + 2*avg2 + avg3) / 7
    out = np.full(n, np.nan)
    out[t3-1:] = uo
    return out

In [ ]:
# Compare hand-rolled vs TA-Lib
ultosc_hand = ultosc_handrolled(high, low, close, t1, t2, t3)
df["ULTOSC_Hand"] = ultosc_hand
valid = ~np.isnan(ultosc_talib) & ~np.isnan(ultosc_hand)
diff = np.abs(ultosc_talib[valid] - ultosc_hand[valid])
print(f"Max diff: {diff.max():.6f}")
df[["Close", "ULTOSC_TA-Lib", "ULTOSC_Hand"]].tail(20)

## Signal Logic (momentum mode)

- **Long entry**: ultosc crosses above upper (40)
- **Short entry**: ultosc crosses below lower (60)
- **Exit**: flip, midpoint, max hold, or stop loss